# Embedding a Knowledge Assistant in a SharePoint Page

This notebook walks through embedding the same Databricks App built in notebook 03 into a SharePoint page, so users can ask questions about the page they are looking at without leaving SharePoint.

The notebook is written for someone new to both Databricks and SharePoint administration. Every step is explicit. Common blockers are called out with their fixes.

### What you end up with

A SharePoint page that shows its normal content (news, announcements, document links, lists) alongside an embedded Knowledge Assistant chat panel. Users see the at-a-glance information they already use, plus a chatbot they can ask questions of. The chatbot can answer questions about the SharePoint page content itself, provided you also ran notebook 04 (SharePoint pages ingestion) so the page text is in the Knowledge Assistant's indexed corpus.

## Prerequisites

Before starting, confirm:

- **Notebook 03 is complete.** You have a Databricks App deployed that wraps the Knowledge Assistant. You know its URL (looks like `https://<APP_NAME>-<workspace-id>.<region>.databricksapps.com`) and you have verified it works when opened in a browser.
- **Optional but strongly recommended: notebook 04 is complete.** This gets SharePoint page content into the Knowledge Assistant's indexed corpus so the chatbot can answer questions about what users are looking at on the page. Without it, the chatbot can answer questions about documents but will not know what the SharePoint page itself says.
- **You have edit permission on the SharePoint page** where you want to embed the chatbot. If you cannot see an "Edit" button at the top-right of the page when you visit it in a browser, you do not have edit permission. Ask your SharePoint admin.

## Placeholders used in this notebook

| Placeholder | Description | Where to get it |
|---|---|---|
| `<APP_URL>` | The Databricks App URL from notebook 03 | The App's detail page in Compute > Apps |
| `<SHAREPOINT_PAGE_URL>` | The SharePoint page you want to embed in | Browser address bar |

## Setup: confirm the Databricks App is reachable in a browser

Before configuring the SharePoint side, confirm the Databricks App from notebook 03 still works.

1. Open the Databricks App URL in a regular browser tab.
2. Sign in if prompted.
3. Type a test question. Confirm you get a streaming response back.
4. Keep this browser tab open. You will refer back to this URL in Step 2.

If the App does not work here, fix it before continuing. The embedded version cannot work if the standalone version does not.

## Step 1: Open the SharePoint page in edit mode

1. In a browser, navigate to the SharePoint page where you want to embed the chatbot (e.g., the Data Governance home page).
2. At the top-right of the page, click **Edit**. The page enters edit mode. You will see a left-side panel of available web parts and a `+` icon between every existing section of content.

If you do not see the Edit button, you do not have edit permission on this page. Stop and ask your SharePoint admin to grant you edit access, or have them follow these steps for you.

## Step 2: Add the Embed web part

SharePoint has a built-in web part called **Embed** that displays an external website inside a panel on the page. We will use this to display the Databricks App.

1. Click the `+` icon at the location on the page where you want the chatbot to appear. A web part picker opens.
2. In the picker's search box, type `Embed`.
3. Click the **Embed** web part.
4. The web part is placed on the page. A configuration panel opens on the right.
5. In the configuration panel, paste your Databricks App URL (the full URL starting with `https://`).
6. The Embed web part will preview the App. You should see your chatbot's UI render in the embed area.

### If the preview shows "This content cannot be embedded"

This is the most common blocker. It means the Databricks App's web server is sending HTTP headers that prevent it from being embedded by another site. This is a security feature called X-Frame-Options or Content-Security-Policy. See the Troubleshooting section at the bottom of this notebook for fixes.

## Step 3: Configure the embedded panel's size

The default Embed web part is small. Resize it to give the chat panel enough room:

1. Click the Embed web part to select it.
2. Look for a settings icon (pencil or gear) on the left side of the web part.
3. Adjust the height. A height of 600-800 pixels works well for a chat interface.
4. SharePoint does not always expose height control directly. If you cannot find a height setting, the Embed web part will scale to the available space in the section it sits in. Place it inside a tall single-column section for best results.

## Step 4: Publish the page

1. At the top-right of the page, click **Publish** (or **Republish** if the page was previously published).
2. SharePoint saves the page and exits edit mode.
3. The page is now live for anyone with read access to the site.

## Step 5: Test the end-user experience

1. From a different user's account (or your own in an incognito browser window), open the SharePoint page.
2. Confirm the page loads with the chatbot panel visible.
3. Ask a question. Confirm:
   - The chat panel responds with streaming text.
   - Citations to source documents appear if your Knowledge Assistant is configured to return them.
   - If you ran notebook 04, the chatbot can answer questions about the content of the SharePoint page itself (e.g., "what does the announcement at the top of this page say?").
4. Test from multiple users to confirm no per-user Databricks login prompts appear. (If they do, the Databricks App is not configured to use service principal auth. Revisit notebook 03 Step 4.)

At this point the chatbot is fully integrated into SharePoint. Users see the same page they always used, plus the chatbot embedded inline.

## Troubleshooting

### "This content cannot be embedded" appears in the Embed web part

The Databricks App's HTTP headers are blocking the iframe. There are three possible fixes, in order of preference.

#### Fix 1: Confirm the App URL works in a browser tab first

Open the App URL in a new tab. If you get a Databricks login prompt, the App is not configured to use service principal auth. In that case, SharePoint cannot embed it either because every user would need to sign in inside the iframe (and SharePoint typically blocks that). Fix the App configuration first (notebook 03 Step 4).

#### Fix 2: Adjust the Databricks App to send permissive iframe headers

By default, Streamlit on Databricks Apps may emit security headers that prevent embedding. Edit your `app.py` from notebook 03 to inject permissive headers via a small wrapper. Add the following at the top of `app.py` after the existing imports, BEFORE the `st.set_page_config(...)` call:

```python
import streamlit.web.server.server as st_server

_original_set_headers = st_server.Server._set_security_headers

def _permissive_security_headers(self, headers):
    _original_set_headers(self, headers)
    # Allow embedding from your SharePoint tenant. Replace with your actual tenant.
    headers['Content-Security-Policy'] = (
        "frame-ancestors 'self' https://<TENANT_SUBDOMAIN>.sharepoint.com"
    )
    headers.pop('X-Frame-Options', None)

st_server.Server._set_security_headers = _permissive_security_headers
```

Replace `<TENANT_SUBDOMAIN>` with your actual SharePoint tenant subdomain (e.g., `acme-corp` if your SharePoint URL starts with `https://acme-corp.sharepoint.com/`).

Then redeploy the Databricks App and re-test the SharePoint embed.

**Caveat:** this monkey-patches a Streamlit internal API. The internal API may change between Streamlit versions. If it does, you will need to find the new equivalent. A more durable fix is described in Fix 3.

#### Fix 3: Rebuild the App as FastAPI + plain HTML

If Fix 2 is too fragile for production, rebuild the chat App using FastAPI with explicit response headers. FastAPI lets you set arbitrary HTTP headers cleanly without monkey-patching anything.

This is more work (you have to build the chat UI as plain HTML and JavaScript instead of using Streamlit's chat primitives), but it is the most reliable embedding pattern. The Databricks Apps documentation has a FastAPI quickstart that demonstrates the structure.

### Page sign-in loop when loading the SharePoint page

When the Embed web part renders the iframe, the user is asked to sign in again, even though they are already signed into SharePoint.

- **Cause:** the Databricks App is configured to require user-level auth instead of service principal auth.
- **Fix:** revisit notebook 03 Step 4. The App's service principal must have `CAN_QUERY` on the Knowledge Assistant endpoint, and `DATABRICKS_TOKEN` must be auto-injected by the Apps runtime. End users should never see a Databricks login screen.

### Chatbot loads but answers are wrong or empty for page-content questions

The user asks "what is this page about?" or "summarize the announcement at the top" and the chatbot does not know.

- **Cause:** the SharePoint page content is not in the Knowledge Assistant's indexed corpus.
- **Fix:** run notebook 04 (SharePoint pages ingestion) to land the page content in a Delta table, then point Knowledge Assistant at that table as an additional knowledge source.

### First load is very slow

The Databricks App takes 10-30 seconds to respond on the first request after a period of inactivity.

- **Cause:** Databricks Apps scale to zero by default.
- **Fix:** see notebook 03 Step 3's "Performance: keep the app warm" section. Set the App's minimum instances to 1, or schedule a keep-alive ping.

## When the iframe approach is not enough

The Embed web part works for most use cases. Two more sophisticated alternatives exist if you need a fully native SharePoint experience or if iframe headers consistently block the embed:

### Alternative 1: SharePoint Framework (SPFx) web part

A SPFx web part is a custom React component packaged for SharePoint. It is treated as a first-class SharePoint component (not embedded via iframe), so it has none of the iframe header issues. The web part renders inline on the page and looks indistinguishable from other native SharePoint web parts.

- **Effort:** 1-2 weeks of SPFx development. Requires a developer at your organization who knows SharePoint Framework, plus a Microsoft 365 dev tenant for development and a SharePoint app catalog for tenant-wide deployment.
- **When to choose:** the iframe approach works but stakeholders want a more polished, native-feeling experience for production rollout.

### Alternative 2: Power Apps embedded

Power Apps is Microsoft's drag-and-drop app builder. Build a Power App with a chat canvas that uses a custom connector to call the Knowledge Assistant serving endpoint. Embed the Power App on the SharePoint page using the Power Apps web part.

- **Effort:** 1-2 days if someone at your organization knows Power Platform. Requires Power Apps licensing (premium for custom connectors).
- **When to choose:** you already use Power Platform extensively and want a Microsoft-managed front-end without writing SPFx code.

Both alternatives still call the same Databricks Knowledge Assistant serving endpoint. Only the consumption surface changes.

## References

- [SharePoint Embed web part](https://support.microsoft.com/en-us/office/use-the-embed-web-part-1providing-modern-experience-pages-acff6d39-2d22-432c-9bb4-322a0cdfeed5)
- [Knowledge Assistant](https://docs.databricks.com/aws/en/generative-ai/agent-bricks/knowledge-assistant)
- [Databricks Apps overview](https://docs.databricks.com/aws/en/dev-tools/databricks-apps/)
- [Content-Security-Policy: frame-ancestors](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/Content-Security-Policy/frame-ancestors)
- [Building SPFx web parts](https://learn.microsoft.com/en-us/sharepoint/dev/spfx/web-parts/get-started/build-a-hello-world-web-part)